This script will guide you through the steps to test and evaluate the accuracy of the trained model. More specifically, it prepares the padded png image for manual labelling, segments the labelled image, runs the detection on the same non-labelled image and backward annotates it, compares your manual annotation to the detected annotation, and renders a confusion matrix to evaluate the model performance. 

#### 1. Paths definition and set-up

Before setting all your paths up and running code, make sure you have set up your "config_test.yaml" config file. You will find specific guidance directly in the file on how exactly the set up needs to be but paths are all relative and should not need to be modified. Make sure to double check nevertheless.   
Once that is done, you can proceed here with the coding section.   
Here bellow, you only need to adjust the RUN_FOLDER to your folder of choice containing the images to test your trained model. 

In [ ]:
import os
import shutil
import yaml 
import argparse
import os.path as path
import scipy.cluster
import scipy.spatial
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import scipy
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from tqdm import tqdm
import torch
import stat
import shutil
from datetime import datetime

from counting_boats.boat_utils.config import cfg
from counting_boats.boat_utils import image_cutting_support as ics
from counting_boats.boat_utils import heatmap as hm

import counting_boats.boat_utils.classifier
# cluster, process_clusters, read_classifications, pixel2latlong

# Every path in this notebook is relative to the REPOSITORY ROOT, so that the
# same notebook works for everyone on macOS, Windows and Linux. This notebook
# lives at the repo root, so Jupyter should already start here - the check below
# turns a wrong working directory into a clear message instead of a confusing
# "file not found" further down.
REPO_ROOT = os.path.abspath(".")
assert os.path.isdir(os.path.join(REPO_ROOT, "counting_boats")), (
    f"Run this notebook from the repository root (the folder containing "
    f"counting_boats/). Current working directory: {REPO_ROOT}"
)
sys.path.insert(0, REPO_ROOT)

#---------------------------------------# 
# Modify the RUN_FOLDER here bellow: 
# Run folder and config used by every step below.
# RUN_FOLDER must match 'path:' in the config file.

RUN_FOLDER = "./testing"
CONFIG = "config_test.yaml"
print(f"Repo root:   {REPO_ROOT}")
print(f"Run folder:  {RUN_FOLDER}")
print(f"Config:      {CONFIG}")

#### 2. Preparation of the padded PNG image
Using a testing .tif file image of your choice, defined in the RUN_FOLDER, this steps prepares the padded png image for future stages.  
To use a .tif file obtained from Planet, use the function testing.prepare(), whereas if you are using a .tif file from Sentinel, you should use the testing.prepare_S2() function. 

In [ ]:
# Run preparation: TIFF -> padded PNG, ready for labelling in labelme.
# Reads the .tif files from <RUN_FOLDER>/<raw_images> and writes PNGs to
# <RUN_FOLDER>/<pngs>, both defined in the config.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.prepare(RUN_FOLDER, CONFIG)

#### 3. Manual annotation 
Annotate manually the created png with labelme as in the training step. Refer to the testing tab of this tutorial for detailed instructions on how to use labelme software. Make sure to use the same categories as the training step. This will allow us to compare my annotation to the detection of the trained model, i.e. test the model. 

#### 4. Segmentation of the annotated image
Once the manual annotation is done, we can apply the segmentation. Note that this segmentation doesn't split 20% of the images for the model validation, meaning all segmented images will be used for the testing. 

In [ ]:
# Run segmentation WITHOUT splitting off 20% for validation
# (this is test data, not training data - we want every tile).
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.segment(RUN_FOLDER, CONFIG)

#### 5. Waterholes detection with the trained model
Using those segmented labelled images, we can run the detection of waterholes using the trained model, and compare our manual labelling with the detection of the model. 

Debugging section to recognise and work well with the GPU:

Note to user: Need to update the torchvision to match the cuda (GPU) version. using the 'nvidia-smi' command, you get the cuda version (my case: 11) so I need to get a version of torch and torchaudio with to 11.xx. Need to 'pip uninstall torch torchvision', and then install the correct version, in my case: 'pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113'  
Other version to be found on this website: https://pytorch.org/get-started/previous-versions/

The testing.segment function and run_detection work to use the segmented images folders grouped per date. left as it is for now but just something to bear in mind!  

In [ ]:
#check of the GPU found or not?
torch.cuda.is_available()

In [ ]:
# Run detection on the testing tiles.
# Uses the weights and device set in the config (device: auto picks
# cuda / mps / cpu for whichever machine you are on).
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.run_detection(RUN_FOLDER, CONFIG)

Copy of the order command that is run run to yolo cmd  by the function run_detectionpage. Just in order to debug and be informative. Do not run.

In [ ]:
# REFERENCE ONLY - this is the command run_detection() builds and runs for you.
# Kept here for debugging; it is commented out because it is shell syntax, not
# Python, and would raise a SyntaxError if executed.
#
# python <yolo_dir>/detect.py --imgsz 416 --save-txt --save-conf #     --weights ./data/NN_weights.pt #     --source ./testing/segmented_images/<date>/<image_name> #     --device <auto-resolved> --nosave --conf-thres 0.15
#
# To see the exact command for your machine, run the cell above - it prints it.

#### 6. Backward annotation 
We can now use the detection output of the model on my testing images to produce labelme style annotation.   

Use the detection output of the model on my testing images to produce labelme style annotation using the backwards_annotation():  

In [ ]:
# Turn the model detections into labelme-style annotations, so they can be
# opened next to the manual labels for visual comparison.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.backwards_annotation_AF(RUN_FOLDER, CONFIG)

#### 7. Ground truth comparison
Then finally, we can compare the model-detected waterholes with the ones you manually labeled. 

In [ ]:
# Compare my labels with the detected waterholes.
# Writes one comparison CSV per image into RUN_FOLDER.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.compare_detections_to_ground_truth(RUN_FOLDER, CONFIG)

#### 8. Confusion matrix
Create the confusion matrix which summaries the performance of the model.

In [ ]:
# Create the confusion matrix from the comparison CSVs in RUN_FOLDER.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.confusion_matrix_AF(RUN_FOLDER, CONFIG)

### Model assessment

The confusion matrix above reports overall accuracy only, which lumps *"missed the waterhole entirely"* together with *"found it but called it the wrong type"* — two problems with completely different fixes. The three steps below unpack that.

All of them read the `*.details.csv` files written by `compare_detections_to_ground_truth`. They only read its output, so they are cheap to re-run while tuning.

#### 9. Per-class precision, recall and F1

Splits the result into two questions:

- **Detection** — did we find the waterhole at all, ignoring class? Poor recall means the model is blind to waterholes, or the clustering is dropping them.
- **Classification** — given that we found it, did we get the type right? Computed over matched pairs only, so it is not contaminated by misses.

Then the same broken down per class. Watch the `support` column: a class with few labelled examples gives an unreliable score.

In [ ]:
from counting_boats.boat_utils import evaluation

report = evaluation.classification_report_AF(RUN_FOLDER, CONFIG)

#### 10. Localisation quality and threshold sensitivity

How *tight* the boxes are, not just whether something was found.

- The **overlap distribution** of matched pairs. Values bunched just above `COMPARE_IOU_THRESHOLD` mean the boxes are barely qualifying and localisation is the weak point. Values up around 0.8+ mean tight boxes.
- The **threshold sweep** shows how fast matches disappear as you demand tighter overlap.

This is how you set `COMPARE_IOU_THRESHOLD` and the per-class `class_iou_threshold` values on evidence — they are currently placeholders of 0.5.

Note the sweep reports localisation *headroom*, not a recommended threshold: a lower threshold can only add matches, so F1 is always highest at the smallest threshold and cannot be used to choose one.

In [ ]:
from counting_boats.boat_utils import evaluation

summary, sweep = evaluation.localisation_quality_AF(RUN_FOLDER, CONFIG)

# To sweep a custom range instead:
# summary, sweep = evaluation.localisation_quality_AF(
#     RUN_FOLDER, CONFIG, thresholds=[0.3, 0.4, 0.5, 0.6, 0.7]
# )

#### 11. Recall by waterhole size

Are the small waterholes being missed? An aggregate recall of 0.85 can hide 0.95 on large waterholes and 0.40 on small ones — invisible in every other metric here, and it would change how the ecological results should be read.

Sizes come from the ground-truth box area in pixels, split into five equal-count buckets by default so every bucket has a usable sample.

In [ ]:
from counting_boats.boat_utils import evaluation

size_table = evaluation.recall_by_size_AF(RUN_FOLDER, CONFIG)

# Add by_class=True to break it down per waterhole type as well,
# or pass explicit area bin edges in square pixels:
# size_table = evaluation.recall_by_size_AF(
#     RUN_FOLDER, CONFIG, bins=[0, 400, 1600, 6400, 25600, 1e9], by_class=True
# )

### Extra testing outputs 
I. Possible to process a single images by comparing the detections and labels for a single image.

Possible to process a single images by comparing the detections and labels for a single image, used in a function but not usefull by hand. 

In [ ]:
# Optional: process a single image, comparing its detections against its labels.
# Useful for debugging one scene rather than the whole run.
# import counting_boats.boat_utils.testing
#
# counting_boats.boat_utils.testing.process_image_AF(
#     os.path.join(RUN_FOLDER, "classifications"),
#     os.path.join(RUN_FOLDER, "labels"),
#     CONFIG,
# )

Compare the counts one by one:

In [ ]:
# Compare the counts one by one: labelled vs detected waterholes.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.waterholes_count_compare(RUN_FOLDER, CONFIG)

#### Global image generation
Now we want to plot the waterholes on the images. To do that, I need first to stich the training images together:

In [ ]:
# Stitch the segmented tiles back into whole images before plotting.
import os
import datetime
from PIL import Image
import counting_boats.boat_utils.stitch_PNGs

counting_boats.boat_utils.stitch_PNGs.stitch(os.path.join(RUN_FOLDER, "stitching"))

Now I want to plot the waterholes, but the function is made for boats so I need to modify it to work on WH. WIP

In [ ]:
# Plot the waterholes on the stitched images.
# First argument is the folder holding the comparison CSVs written by
# compare_detections_to_ground_truth() above, i.e. RUN_FOLDER.
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.plot_waterholes(RUN_FOLDER, CONFIG)

---
## Visual review — labelled vs detected

A **4 x 4 grid of 16 waterholes per page**, each cell zoomed in on one waterhole:

- **solid box** = your label, coloured by the class you assigned
- **dashed box** = the model detection, coloured by the class it predicted

Two different colours in a cell means the classes disagree — visible at a glance without reading anything. A dashed box on its own is a false positive; a solid box on its own is a waterhole the model missed. The legend down the right side carries the class colours, the box-style key, and live counts.

Page with the **arrow keys**. Filter with **`e`** (errors only), **`m`** (class disagreements), **`f`** (false positives), **`x`** (missed), **`a`** (all). **`o`** cycles the sort, **`s`** saves the page as a PNG, **`q`** closes. Full list is printed when it opens.

Needs `compare_detections_to_ground_truth` to have run first — it reads the `*.details.csv` files that step writes. It changes nothing, so it is safe to re-run.

In [ ]:
%matplotlib qt
from counting_boats.boat_utils import review

reviewer = review.launch(RUN_FOLDER, CONFIG)

The window opens on everything, in image order. To start straight on the problem cases, or to change the grid, pass a `ReviewParams`:

In [ ]:
# Optional variants -- uncomment whichever you want.

# Straight to the errors, worst overlap first:
# reviewer = review.launch(RUN_FOLDER, CONFIG, review.ReviewParams(
#     filter="errors", sort="worst_overlap"))

# Bigger cells (3 x 3 = 9 per page) for a closer look:
# reviewer = review.launch(RUN_FOLDER, CONFIG, review.ReviewParams(
#     panel_rows=3, panel_cols=3, crop_margin=2.0))

# Only one class:
# reviewer = review.launch(RUN_FOLDER, CONFIG, review.ReviewParams(
#     only_classes=("WH_swamp",)))

# What is on screen right now, as a DataFrame:
# reviewer.to_dataframe().head(20)

# No display available (headless / for a report) -- write a page straight to PNG:
# review.save_contact_sheet(RUN_FOLDER, CONFIG,
#                           review.ReviewParams(filter="errors"), page=0)